In [ ]:
# pip install insightface onnxruntime-gpu opencv-python numpy matplotlib Pillow
import random
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple
from insightface.app import FaceAnalysis
import matplotlib.pyplot as plt
from scipy import ndimage

@dataclass
class FaceInfo:
    embedding: np.ndarray
    det_score: float
    bbox: np.ndarray
    kps: Optional[np.ndarray] = None

class FaceNormalizer:
    def __init__(self, target_size=(112, 112)):
        self.target_size = target_size
        # 標準的な顔のランドマーク位置（正規化後の目と口の位置）
        self.target_landmarks = np.array([
            [0.35, 0.35],  # 左目
            [0.65, 0.35],  # 右目
            [0.5, 0.7],    # 口
        ]) * target_size[0]

    def normalize_face(self, img: np.ndarray, landmarks: np.ndarray) -> np.ndarray:
        """顔画像の正規化（アライメント、照明補正、ガンマ補正）"""
        # ランドマークから3点を選択（左目、右目、口）
        src_points = landmarks[[0, 1, 3]].astype(np.float32)
        dst_points = self.target_landmarks.astype(np.float32)
        
        # アフィン変換行列を計算
        M = cv2.getAffineTransform(src_points, dst_points)
        aligned = cv2.warpAffine(img, M, self.target_size)
        
        # 照明補正（CLAHE）
        lab = cv2.cvtColor(aligned, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        lab[...,0] = clahe.apply(lab[...,0])
        normalized = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        
        # ガンマ補正
        gamma = 0.8  # 暗い部分を明るく
        normalized = np.clip((normalized / 255.0) ** gamma * 255.0, 0, 255).astype(np.uint8)
        
        return normalized

class FaceRecognizer:
    def __init__(self, model_root: str = "../models"):
        self.app = FaceAnalysis(
            name='antelopev2/antelopev2',
            root=model_root,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
            allowed_modules=['detection', 'recognition']
        )
        self.app.prepare(ctx_id=0, det_size=(640, 640))
        self.normalizer = FaceNormalizer()
        self.scales = [0.8, 1.0, 1.2]  # マルチスケール処理用

    def get_multi_scale_embedding(self, img_bgr: np.ndarray, face: FaceInfo) -> np.ndarray:
        """マルチスケールでの特徴量抽出"""
        if face.kps is None or face.embedding is None:
            return None
            
        embeddings = []
        # 基本の正規化画像からの特徴量
        norm_img = self.normalizer.normalize_face(img_bgr, face.kps)
        embeddings.append(face.embedding)
        
        # スケール変化させた画像からの特徴量
        h, w = norm_img.shape[:2]
        for scale in self.scales:
            if scale == 1.0:
                continue
            scaled = cv2.resize(norm_img, (int(w*scale), int(h*scale)))
            scaled = cv2.resize(scaled, (w, h))  # 元のサイズに戻す
            faces = self.app.get(scaled)
            if faces:
                embeddings.append(faces[0].normed_embedding)
        
        # 複数の特徴量を平均して正規化
        if embeddings:
            avg_embedding = np.mean(embeddings, axis=0)
            return avg_embedding / np.linalg.norm(avg_embedding)
        return None

    def detect_faces(self, img_bgr: np.ndarray) -> List[FaceInfo]:
        faces = self.app.get(img_bgr)
        return [FaceInfo(
            embedding=self.get_multi_scale_embedding(img_bgr, f),
            det_score=f.det_score,
            bbox=f.bbox,
            kps=getattr(f, 'kps', None)
        ) for f in faces]

    def embed(self, img_bgr: np.ndarray) -> Optional[np.ndarray]:
        faces = self.detect_faces(img_bgr)
        if not faces:
            return None
        return max(faces, key=lambda x: x.det_score).embedding

class Gallery:
    def __init__(self, recognizer: FaceRecognizer):
        self.data: Dict[str, List[np.ndarray]] = {}
        self.recognizer = recognizer

    def add_face(self, person_id: str, img_bgr: np.ndarray) -> bool:
        embedding = self.recognizer.embed(img_bgr)
        if embedding is None:
            return False
        self.data.setdefault(person_id, []).append(embedding)
        return True

    def get_person_vector(self, person_id: str) -> Optional[np.ndarray]:
        vecs = self.data.get(person_id, [])
        if not vecs:
            return None
        v = np.mean(np.stack(vecs, axis=0), axis=0)
        return v / (np.linalg.norm(v) + 1e-12)

    def recognize(self, img_bgr: np.ndarray, threshold: float = 0.35) -> Tuple[Optional[str], float]:
        query_embedding = self.recognizer.embed(img_bgr)
        if query_embedding is None:
            return None, 0.0

        best_id, best_score = None, -1.0
        for pid in self.data:
            gallery_vec = self.get_person_vector(pid)
            if gallery_vec is None:
                continue
            score = cosine_sim(query_embedding, gallery_vec)
            if score > best_score:
                best_id, best_score = pid, score
        return (best_id, best_score) if best_score >= threshold else (None, best_score)

def get_bgr_img(path: Path) -> np.ndarray:
    try:
        pil_image = Image.open(str(path)).convert("RGB")
        return cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    except Exception as e:
        raise ValueError(f"Failed to read image {path}: {e}")

def draw_faces(img_bgr: np.ndarray, faces: List[FaceInfo]) -> np.ndarray:
    out = img_bgr.copy()
    for face in faces:
        x1, y1, x2, y2 = face.bbox.astype(int)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        if face.kps is not None:
            for (x, y) in face.kps.astype(int):
                cv2.circle(out, (x, y), 2, (255, 0, 0), -1)
        cv2.putText(out, f"{face.det_score:.2f}", 
                    (x1, max(0, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1, cv2.LINE_AA)
    return out

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def create_comparison_image(img_bgr: np.ndarray, face1: FaceInfo, face2: FaceInfo, score: float) -> np.ndarray:
    # 顔領域の切り出し
    def crop_face(face: FaceInfo, margin: float = 0.2) -> np.ndarray:
        x1, y1, x2, y2 = face.bbox.astype(int)
        h, w = y2 - y1, x2 - x1
        margin_px = int(max(h, w) * margin)
        y1, y2 = max(0, y1 - margin_px), min(img_bgr.shape[0], y2 + margin_px)
        x1, x2 = max(0, x1 - margin_px), min(img_bgr.shape[1], x2 + margin_px)
        return img_bgr[y1:y2, x1:x2]

    face1_img = crop_face(face1)
    face2_img = crop_face(face2)

    # リサイズして統一サイズに
    target_size = (200, 200)
    face1_resized = cv2.resize(face1_img, target_size)
    face2_resized = cv2.resize(face2_img, target_size)

    # 横に並べて余白を追加
    margin = 100
    height = target_size[1]
    width = target_size[0] * 2 + margin
    combined = np.ones((height, width, 3), dtype=np.uint8) * 255

    # 画像配置
    combined[:, :target_size[0]] = face1_resized
    combined[:, -target_size[0]:] = face2_resized

    # スコアテキスト描画
    text = f"{score:.3f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0
    thickness = 2
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    text_x = (width - text_size[0]) // 2
    text_y = height // 2
    cv2.putText(combined, text, (text_x, text_y), font, font_scale, (0, 0, 0), thickness)

    return combined

def main():
    recognizer = FaceRecognizer()
    gallery = Gallery(recognizer)
    
    # 画像を1枚ランダム取得
    candidates = list(Path('../data/欅坂46/[20190517] B.L.T graph 武本唯衣/').glob("**/*.png"))
    if not candidates:
        raise FileNotFoundError("画像が見つかりませんでした。'../data' に .jpg を配置してください。")
    image_path = random.choice(candidates)
    print(f"[INFO] image_path: {image_path}")

    img_bgr = get_bgr_img(image_path)

    # 顔検出（= 整列＋埋め込み計算まで行われる）
    faces = recognizer.detect_faces(img_bgr)
    if not faces:
        print("No faces found.")
        return

    # Notebook可視化：枠とランドマークを描画して表示
    vis_bgr = draw_faces(img_bgr, faces)
    vis_rgb = cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2RGB)
    
    pil_img = Image.fromarray(vis_rgb)
    display(pil_img)

    # # それぞれの顔の特徴量（512D）をそのままprint
    # np.set_printoptions(suppress=True, precision=5, linewidth=200)
    # for i, face in enumerate(faces):
    #     emb = face.embedding  # L2正規化済み 512次元
    #     if emb is None:
    #         print(f"[Face {i}] embedding: None (認識モジュールがロードされていない可能性)")
    #     else:
    #         print(f"[Face {i}] embedding shape: {emb.shape}, dtype: {emb.dtype}")
    #         print(emb)  # 要求通り“そのまま”出力

    # （参考）従来のデモ：自身をギャラリーに登録して照合
    if gallery.add_face("person1", img_bgr):
        recognized_id, score = gallery.recognize(img_bgr)
        print(f"Recognized ID: {recognized_id}, Score: {score:.4f}")
    else:
        print("Failed to add face to gallery")

    # 顔の組み合わせごとの類似度を計算して表示
    if len(faces) >= 2:
        print("\n=== Face Similarity Matrix ===")
        fig = plt.figure(figsize=(15, len(faces) * 2))
        plot_idx = 1
        
        for i, face1 in enumerate(faces):
            for j, face2 in enumerate(faces[i+1:], i+1):
                score = cosine_sim(face1.embedding, face2.embedding)
                print(f"Face {i} vs Face {j}: {score:.3f}")
                
                comparison = create_comparison_image(img_bgr, face1, face2, score)
                plt.subplot(len(faces)-1, len(faces)-1, plot_idx)
                plt.imshow(cv2.cvtColor(comparison, cv2.COLOR_BGR2RGB))
                plt.axis('off')
                plt.title(f'Face {i} vs Face {j}')
                plot_idx += 1
        
        plt.tight_layout()
        plt.show()
    else:
        print("Need at least 2 faces for comparison")

if __name__ == "__main__":
    main()

In [ ]:
import shutil
from pathlib import Path
from sklearn.cluster import DBSCAN
from collections import defaultdict
from tqdm import tqdm

def create_face_image(img_bgr: np.ndarray, face: FaceInfo, margin: float = 0.2) -> np.ndarray:
    """顔領域を切り出して正規化"""
    # 顔領域の切り出し
    x1, y1, x2, y2 = face.bbox.astype(int)
    h, w = y2 - y1, x2 - x1
    margin_px = int(max(h, w) * margin)
    
    y1 = max(0, y1 - margin_px)
    y2 = min(img_bgr.shape[0], y2 + margin_px)
    x1 = max(0, x1 - margin_px)
    x2 = min(img_bgr.shape[1], x2 + margin_px)
    
    face_img = img_bgr[y1:y2, x1:x2].copy()
    
    # 正規化された顔画像を取得
    if face.kps is not None:
        # ランドマークの座標を切り出し領域に合わせて調整
        kps = face.kps.copy()
        kps[:, 0] -= x1
        kps[:, 1] -= y1
        normalizer = FaceNormalizer()
        face_img = normalizer.normalize_face(face_img, kps)
        
    return face_img

class FaceClustering:
    def __init__(self, output_dir: Path, min_faces: int = 2, threshold: float = 0.6):
        self.recognizer = FaceRecognizer()
        self.output_dir = output_dir
        self.min_faces = min_faces
        self.threshold = threshold
        self.faces_dir = output_dir / "faces"
        self.clusters_dir = output_dir / "clusters"
        self.faces_dir.mkdir(parents=True, exist_ok=True)
        self.clusters_dir.mkdir(parents=True, exist_ok=True)
        self.source_images: Dict[Path, Path] = {}  # 顔画像と元画像のマッピング

    def extract_faces(self, input_dir: Path) -> List[Tuple[Path, FaceInfo]]:
        """画像から顔を抽出して保存"""
        faces_list = []
        face_count = 0
        
        image_files = list(input_dir.glob("**/*.jpg")) + list(input_dir.glob("**/*.png"))
        for img_path in tqdm(image_files, desc="Extracting faces"):
            try:
                img_bgr = get_bgr_img(img_path)
                faces = self.recognizer.detect_faces(img_bgr)
                
                for i, face in enumerate(faces):
                    if face.embedding is None:
                        continue
                    
                    # 顔画像を切り出して保存
                    face_img = create_face_image(img_bgr, face)
                    face_filename = f"{img_path.stem}_{i}.png"
                    face_path = self.faces_dir / face_filename
                    cv2.imwrite(str(face_path), face_img)
                    
                    # 元画像とのマッピングを保存
                    self.source_images[face_path] = img_path
                    
                    faces_list.append((face_path, face))
                    face_count += 1
                    
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                
        print(f"Total faces extracted: {face_count}")
        return faces_list

    def cluster_faces(self, faces_list: List[Tuple[Path, FaceInfo]]) -> Dict[int, List[Path]]:
        """顔をクラスタリング"""
        if not faces_list:
            return {}
            
        # 埋め込みベクトルを準備
        embeddings = np.array([face.embedding for _, face in faces_list])
        
        # DBSCANでクラスタリング
        clustering = DBSCAN(
            eps=1-self.threshold,  # 類似度閾値から距離に変換
            min_samples=self.min_faces,
            metric='cosine'
        ).fit(embeddings)
        
        # クラスタごとにファイルをグループ化
        clusters = defaultdict(list)
        for (face_path, _), label in zip(faces_list, clustering.labels_):
            if label >= 0:  # -1はノイズ
                clusters[label].append(face_path)
                
        return dict(clusters)

    def save_clusters(self, clusters: Dict[int, List[Path]]):
        """クラスタごとにフォルダを作成して顔画像と元画像を保存"""
        for cluster_id, face_paths in clusters.items():
            # クラスタのディレクトリ作成
            cluster_dir = self.clusters_dir / f"person_{cluster_id:03d}"
            faces_dir = cluster_dir / "faces"
            sources_dir = cluster_dir / "sources"
            faces_dir.mkdir(parents=True, exist_ok=True)
            sources_dir.mkdir(exist_ok=True)
            
            # 既にコピーした元画像を記録（重複を避けるため）
            copied_sources = set()
            
            for i, face_path in enumerate(face_paths):
                # 顔画像のコピー
                new_face_path = faces_dir / f"{cluster_id:03d}_{i:03d}{face_path.suffix}"
                shutil.copy2(face_path, new_face_path)
                
                # 元画像のコピー
                source_path = self.source_images[face_path]
                if source_path not in copied_sources:
                    new_source_path = sources_dir / source_path.name
                    shutil.copy2(source_path, new_source_path)
                    copied_sources.add(source_path)
        
        print(f"Saved {len(clusters)} clusters with source images")

def main():
    input_dir = Path("../data/欅坂46/")
    output_dir = Path("../output/test/")
    
    clustering = FaceClustering(
        output_dir=output_dir,
        min_faces=2,
        threshold=0.6
    )
    
    # 顔の抽出
    faces_list = clustering.extract_faces(input_dir)
    
    # クラスタリング
    clusters = clustering.cluster_faces(faces_list)
    
    # 結果の保存
    clustering.save_clusters(clusters)

if __name__ == "__main__":
    main()